# IoU de caja por diente de los 4 segmentadores de la etapa 1 (U-Net++, DeepLabV3+, SegFormer-B2, YOLO11m-seg)
**Settings:** **CPU**, Internet OFF (no instala nada). Sin re-entrenar ni re-inferir.
**Inputs:**
- salida de `benchmark-segmentacion-cv-nino`: `results/<modelo>/fold<k>_preds.npz` (argmax a resolución nativa; YOLO pintado con conf ≥ 0,25) y `results/metricas_ic95.csv`;
- salida de `datasets-limpios-icdas-segmentacion`: `segmentation_manifest.csv` y `fdi_segmentation_curated.zip` (máscaras FDI de referencia, labelmap);
- *(opcional, solo para S2)* salida de `map-segmentadores`: `map/instancias.pkl.gz` (instancias crudas de YOLO en RLE).

Especificación: `claude/encargo_iou_caja.md`. Lectura de máscaras, labelmap Universal→FDI y umbral de 50 px de "clase predicha" copiados del benchmark y del notebook de mAP.

**Regla principal.** Unidad = diente de referencia (foto × FDI presente en la referencia; deben ser 9.710 en 630 fotos de 96 niños).
- Caja de referencia: rectángulo envolvente de todos los píxeles de esa clase en la máscara de referencia (= instancia de referencia del mAP).
- Caja predicha: rectángulo envolvente de todos los píxeles predichos con esa clase, si suman ≥ 50 px; si no, el diente queda **no encontrado** (IoU = 0 en la métrica de punta a punta).
- IoU de caja en píxeles nativos, sin margen. Cajas semiabiertas `[y0, y1) × [x0, x1)`: área = número de píxeles del rectángulo.

**Bootstrap:** 2.000 remuestreos de niños, `default_rng(2026).integers(...)`, los mismos para los 4 modelos y los mismos que en `map-segmentadores`.

In [ ]:
import os, glob, gzip, pickle, time, zipfile, re
import numpy as np, pandas as pd, cv2
from scipy import ndimage
from concurrent.futures import ThreadPoolExecutor
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
MODELS = ['unetpp_r50', 'deeplabv3p_r50', 'segformer_b2', 'yolo11m_seg']
LABEL = {'unetpp_r50': 'U-Net++', 'deeplabv3p_r50': 'DeepLabV3+', 'segformer_b2': 'SegFormer-B2', 'yolo11m_seg': 'YOLO11m-seg'}
COL = dict(zip(MODELS, ['#4C72B0', '#55A868', '#C44E52', '#8172B2']))   # mismos colores que curvas_pr_iou50
MIN_PX = 50                 # umbral de "clase predicha" del benchmark
THRS = (0.5, 0.75, 0.9)
B, SEED = 2000, 2026
TOL = 0.001                 # chequeo contra metricas_ic95.csv
EXPECTED = {'unetpp_r50': (0.873, 0.926), 'deeplabv3p_r50': (0.882, 0.940), 'segformer_b2': (0.901, 0.942), 'yolo11m_seg': (0.871, 0.951)}   # Dice por diente / F1 (Dice ≥ 0,5), del encargo
N_REF_ESPERADO = 9710
S2_CONF = 0.25              # S2: misma confianza mínima que la máscara pintada del benchmark
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
OUT = f'{W}/iou_caja'; os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
T0 = time.time()

## Datos, labelmap y FDI (igual que en el benchmark)
Solo se leen las máscaras de referencia (no las fotos).

In [ ]:
def find(name):
    h = sorted(glob.glob(f'{IN}/**/{name}', recursive=True)); return h[0] if h else None
man = pd.read_csv(find('segmentation_manifest.csv'))
DATA = f'{TMP}/seg'
if not glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True):
    zipfile.ZipFile(find('fdi_segmentation_curated.zip')).extractall(DATA)
ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True)[0]))
LM = (glob.glob(f'{DATA}/**/labelmap.txt', recursive=True) or [find('labelmap.txt')])[0]
lines = [l.strip() for l in open(LM, encoding='utf-8') if l.strip() and not l.startswith('#')]
NAMES, COLORS = [], []
for l in lines:
    p = l.split(':'); NAMES.append(p[0].strip()); COLORS.append(tuple(int(v) for v in p[1].split(',')) if len(p) > 1 and p[1] else None)
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0:
    i = bg[0]; NAMES.insert(0, NAMES.pop(i)); COLORS.insert(0, COLORS.pop(i))
NC = len(NAMES)
m0 = cv2.imread(os.path.join(ROOT, man.mask_file[0]), cv2.IMREAD_UNCHANGED); RGB_MASK = m0.ndim == 3
LUT = {c: i for i, c in enumerate(COLORS) if c is not None}
def read_mask(rel):
    m = cv2.imread(os.path.join(ROOT, rel), cv2.IMREAD_UNCHANGED)
    if not RGB_MASK: return m.astype(np.uint8)
    m = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB); key = (m[..., 0].astype(np.int32) << 16) | (m[..., 1].astype(np.int32) << 8) | m[..., 2]
    out = np.zeros(m.shape[:2], np.uint8); seen = set(np.unique(key))
    for c, i in LUT.items():
        k = (c[0] << 16) | (c[1] << 8) | c[2]
        if k in seen: out[key == k] = i
    return out
with ThreadPoolExecutor(8) as ex:
    MASKS = dict(zip(man.image_id, ex.map(read_mask, man.mask_file)))
def univ(n):
    m = re.search(r'\d+', n); return int(m.group()) if m else None
def to_fdi(u):
    if u is None or not 1 <= u <= 32: return None
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
U = [univ(n) for n in NAMES]; FDI = [to_fdi(u) if i else None for i, u in enumerate(U)]
assert NC == 33 and all(f is not None for f in FDI[1:])
TEETH = list(range(1, NC))
assert man.groupby('child_id').fold.nunique().max() == 1
def grupo(f):
    u = f % 10
    return {1: 'incisivos', 2: 'incisivos', 3: 'caninos', 4: 'premolares', 5: 'premolares', 6: 'molares 1.º–2.º', 7: 'molares 1.º–2.º', 8: 'terceros molares'}[u]
GRUPOS = ['incisivos', 'caninos', 'premolares', 'molares 1.º–2.º', 'terceros molares']
RES = os.path.dirname(find('metricas_ic95.csv')); REF = pd.read_csv(f'{RES}/metricas_ic95.csv').set_index('model')
for m in MODELS:
    for k in range(5): assert os.path.exists(f'{RES}/{m}/fold{k}_preds.npz'), f'falta {RES}/{m}/fold{k}_preds.npz'
print(len(MASKS), 'fotos |', man.child_id.nunique(), 'niños | clases', NC, '| raíz', ROOT, '| predicciones:', RES)
print(man.groupby('fold').agg(fotos=('image_id', 'size'), ninos=('child_id', 'nunique')).T.to_string())
print(REF[['dice_tooth', 'id_f1']].round(3).to_string())

## Cajas por diente (referencia y predichas) desde las predicciones del benchmark
Por foto: `scipy.ndimage.find_objects` da en una pasada la caja de cada clase; `bincount` da sus píxeles. Para S1 se toma, dentro de la caja de la clase, el componente conexo (8-conectividad) más grande. También se acumula la matriz de confusión de píxeles por foto para el chequeo de Dice y F1, y se cuentan las clases predichas (≥ 50 px) ausentes en la referencia (informativo; no entran en la métrica por diente).

In [ ]:
def objs(lbl):
    return np.bincount(lbl.ravel(), minlength=NC)[:NC], ndimage.find_objects(lbl, max_label=NC - 1)
def sl2box(s): return (s[0].start, s[1].start, s[0].stop, s[1].stop)          # (y0, x0, y1, x1), semiabierta
def largest_cc_box(lbl, c, s):
    crop = (lbl[s] == c).astype(np.uint8)
    n, _, st, _ = cv2.connectedComponentsWithStats(crop, connectivity=8)
    q = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = (int(v) for v in st[q, :4]); y0, x0 = s[0].start + y, s[1].start + x
    return (y0, x0, y0 + h, x0 + w), n - 1
def confusion(g, p):
    return np.bincount(g.ravel().astype(np.int64) * NC + p.ravel(), minlength=NC * NC).reshape(NC, NC)

GTB = {}
for j in man.image_id:
    cnt, sl = objs(MASKS[j]); GTB[j] = [(c, sl2box(sl[c - 1]), int(cnt[c])) for c in TEETH if cnt[c] > 0]
n_ref = sum(len(v) for v in GTB.values())
print('dientes de referencia:', n_ref, f'({n_ref / len(GTB):.1f} por foto)')

NA = (-1, -1, -1, -1)
rows, CM, FPC = [], {m: {} for m in MODELS}, {}
for m in MODELS:
    t = time.time(); fp = 0
    for k in range(5):
        P = np.load(f'{RES}/{m}/fold{k}_preds.npz')
        for j in man[man.fold == k].image_id:
            p = P[str(j)]; g = MASKS[j]
            assert p.shape == g.shape, f'{m} {j}: forma {p.shape} ≠ referencia {g.shape}'
            CM[m][j] = confusion(g, p); cnt, sl = objs(p)
            gt_cls = {c for c, _, _ in GTB[j]}; fp += sum(1 for c in TEETH if cnt[c] >= MIN_PX and c not in gt_cls)
            for c, gb, gpx in GTB[j]:
                found = bool(cnt[c] >= MIN_PX)
                if found: pb = sl2box(sl[c - 1]); lb, ncc = largest_cc_box(p, c, sl[c - 1])
                else: pb, lb, ncc = NA, NA, 0
                rows.append((m, j, FDI[c], gpx, *gb, int(cnt[c]), found, *pb, *lb, ncc))
        del P
    FPC[m] = fp; print(f'{LABEL[m]:13s} {time.time() - t:.0f} s')
T = pd.DataFrame(rows, columns=['model', 'image_id', 'fdi', 'gt_px', 'gy0', 'gx0', 'gy1', 'gx1', 'pred_px', 'found',
                                'py0', 'px0', 'py1', 'px1', 'ly0', 'lx0', 'ly1', 'lx1', 'n_comp'])
T = T.merge(man[['image_id', 'child_id', 'fold', 'view', 'condition']], on='image_id', how='left')
T['grupo'] = T.fdi.map(grupo)

def box_iou(a, b):
    """a, b: (n, 4) cajas (y0, x0, y1, x1) semiabiertas."""
    ih = np.clip(np.minimum(a[:, 2], b[:, 2]) - np.maximum(a[:, 0], b[:, 0]), 0, None)
    iw = np.clip(np.minimum(a[:, 3], b[:, 3]) - np.maximum(a[:, 1], b[:, 1]), 0, None)
    inter = ih * iw; aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa + ab - inter)
def area(b): return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
G = T[['gy0', 'gx0', 'gy1', 'gx1']].to_numpy(np.int64)
for pre, out in (('p', ''), ('l', '_S1')):
    Pb = T[[f'{pre}y0', f'{pre}x0', f'{pre}y1', f'{pre}x1']].to_numpy(np.int64)
    T[f'iou{out}'] = np.where(T.found, box_iou(G, Pb), 0.0)
    T[f'razon_area{out}'] = np.where(T.found, area(Pb) / area(G), np.nan)
T['gt_area_caja'] = area(G)
print(len(T), 'filas (diente de referencia × modelo)')

## Chequeos (si el primero falla, el notebook se detiene)
1. Con las mismas predicciones se recalculan el Dice por diente y el F1 de identificación (Dice ≥ 0,5) con las definiciones del benchmark y se comparan con `metricas_ic95.csv` (±0,001) y con los valores del encargo.
2. Dientes de referencia = 9.710 (por modelo) y no encontrados por modelo.

In [ ]:
def bench(m):
    I = np.zeros(NC); S = np.zeros(NC); ident = []; tpp = []
    for j in man.image_id:
        cm = CM[m][j]; inter = np.diag(cm).astype(float); ng = cm.sum(1); npx = cm.sum(0)
        for c in TEETH:
            if ng[c] == 0 and npx[c] < MIN_PX: continue
            d = 2 * inter[c] / max(1, ng[c] + npx[c]); I[c] += inter[c]; S[c] += ng[c] + npx[c]
            if ng[c] > 0: ident.append(d >= 0.5)
            if npx[c] >= MIN_PX: tpp.append(d >= 0.5)
    keep = S[1:] > 0; dice = (2 * I[1:][keep] / S[1:][keep]).mean()
    se, pp = np.mean(ident), np.mean(tpp); return dice, 2 * se * pp / (se + pp)
chk = []
for m in MODELS:
    d, f = bench(m); n = int((T.model == m).sum()); nf = int((~T[T.model == m].found).sum())
    chk.append(dict(modelo=LABEL[m], dice_tooth=d, dice_csv=REF.loc[m, 'dice_tooth'], dice_encargo=EXPECTED[m][0],
                    id_f1=f, id_f1_csv=REF.loc[m, 'id_f1'], id_f1_encargo=EXPECTED[m][1],
                    n_ref=n, no_encontrados=nf, pct_no_encontrados=nf / n, clases_pred_sin_ref=FPC[m]))
chk = pd.DataFrame(chk)
chk['ok_csv'] = ((chk.dice_tooth - chk.dice_csv).abs() <= TOL) & ((chk.id_f1 - chk.id_f1_csv).abs() <= TOL)
chk['ok_encargo'] = ((chk.dice_tooth - chk.dice_encargo).abs() <= TOL + 5e-4) & ((chk.id_f1 - chk.id_f1_encargo).abs() <= TOL + 5e-4)   # el encargo trae 3 decimales
chk['ok_n_ref'] = chk.n_ref == N_REF_ESPERADO
chk.to_csv(f'{OUT}/iou_caja_chequeos.csv', index=False); print(chk.round(4).to_string(index=False))
assert chk.ok_csv.all() and chk.ok_encargo.all(), 'LAS PREDICCIONES NO REPRODUCEN Dice/F1 DEL BENCHMARK: se detiene (ver iou_caja_chequeos.csv)'
assert chk.ok_n_ref.all(), f'el número de dientes de referencia no es {N_REF_ESPERADO}: se detiene'
print(f'chequeos OK: Dice y F1 reproducidos a ±{TOL}; {N_REF_ESPERADO} dientes de referencia en {len(GTB)} fotos de {man.child_id.nunique()} niños')

## Métricas principales: IC 95 % por bootstrap de niños y media ± DE por fold
- `iou_media`: **principal**, de punta a punta (no encontrado = 0).
- `iou_media_enc`, `iou_mediana_enc`: entre los encontrados (con `n_encontrados`).
- `p050`, `p075`, `p090`: proporción de dientes de referencia con IoU de caja ≥ 0,5 / 0,75 / 0,9 (no encontrado cuenta como fallo).
IC para las medias y proporciones (sumas por niño × multiplicidad del remuestreo; equivale a concatenar los niños remuestreados). La mediana se informa sin IC.

In [ ]:
KIDS = np.array(sorted(man.child_id.unique())); KI = {c: i for i, c in enumerate(KIDS)}
rng = np.random.default_rng(SEED)
DRAWS = rng.integers(0, len(KIDS), size=(B, len(KIDS)))       # idéntico a map-segmentadores
WK = np.stack([np.bincount(r, minlength=len(KIDS)) for r in DRAWS])
MET = ['iou_media', 'iou_media_enc', 'p050', 'p075', 'p090']

def mets(d, iou='iou', found='found'):
    x = d[iou].to_numpy(float); f = d[found].to_numpy(bool)
    r = dict(n_ref=len(d), n_encontrados=int(f.sum()), iou_media=x.mean(), iou_media_enc=x[f].mean() if f.any() else np.nan,
             iou_mediana_enc=np.median(x[f]) if f.any() else np.nan)
    for t in THRS: r[f'p{int(round(t * 100)):03d}'] = (x >= t).mean()
    return r
def boot(d, iou='iou', found='found'):
    x = d[iou].to_numpy(float); f = d[found].to_numpy(bool)
    s = pd.DataFrame(dict(kid=d.child_id.map(KI).to_numpy(), n=1, x=x, f=f.astype(int), xf=x * f,
                          **{f'g{int(round(t * 100)):03d}': (x >= t).astype(int) for t in THRS})).groupby('kid').sum().reindex(range(len(KIDS)), fill_value=0)
    n = WK @ s.n.to_numpy(); nf = WK @ s.f.to_numpy()
    with np.errstate(invalid='ignore', divide='ignore'):
        return pd.DataFrame(dict(iou_media=WK @ s.x.to_numpy() / n, iou_media_enc=WK @ s.xf.to_numpy() / nf,
                                 **{f'p{int(round(t * 100)):03d}': WK @ s[f'g{int(round(t * 100)):03d}'].to_numpy() / n for t in THRS}))
BOOT, res, pf_rows = {}, [], []
for m in MODELS:
    d = T[T.model == m]; pt = mets(d); bs = boot(d); BOOT[m] = bs
    pf = pd.DataFrame([dict(fold=k, **mets(x)) for k, x in d.groupby('fold')]); pf_rows += [dict(modelo=LABEL[m], **r) for r in pf.to_dict('records')]
    row = dict(modelo=LABEL[m], **pt)
    for c in MET:
        row[f'{c}_lo'] = bs[c].quantile(.025); row[f'{c}_hi'] = bs[c].quantile(.975)
        row[f'{c}_fold_media'] = pf[c].mean(); row[f'{c}_fold_de'] = pf[c].std()
    res.append(row)
res = pd.DataFrame(res); res.to_csv(f'{OUT}/iou_caja_resumen.csv', index=False)
pd.DataFrame(pf_rows).to_csv(f'{OUT}/iou_caja_por_fold.csv', index=False)
print('IoU de caja por diente — combinado fuera de fold [IC 95 %] | por fold media ± DE')
for _, r in res.iterrows():
    print(f'\n{r.modelo}  (n_ref {r.n_ref}, encontrados {r.n_encontrados}, mediana entre encontrados {r.iou_mediana_enc:.3f})')
    for c in MET: print(f'   {c:14s} {r[c]:.3f} [{r[c + "_lo"]:.3f}–{r[c + "_hi"]:.3f}] | {r[c + "_fold_media"]:.3f} ± {r[c + "_fold_de"]:.3f}')

## Comparación pareada (todos los pares): IoU medio de punta a punta y proporción ≥ 0,75
Diferencia a − b con IC por bootstrap pareado de niños (mismos remuestreos); p bilateral = 2·min(P(Δ ≤ 0), P(Δ ≥ 0)); Holm dentro de cada métrica (6 pares). Igual que `comparacion_pareada_map.csv`.

In [ ]:
cmp = []
for x in ('iou_media', 'p075'):
    part = []
    for a in range(4):
        for b_ in range(a + 1, 4):
            ma, mb = MODELS[a], MODELS[b_]; dl = BOOT[ma][x].to_numpy() - BOOT[mb][x].to_numpy()
            part.append(dict(metrica=x, a=LABEL[ma], b=LABEL[mb], dif=res.set_index('modelo').loc[LABEL[ma], x] - res.set_index('modelo').loc[LABEL[mb], x],
                             lo=np.quantile(dl, .025), hi=np.quantile(dl, .975), p=min(1, 2 * min((dl <= 0).mean(), (dl >= 0).mean()))))
    part = pd.DataFrame(part).sort_values('p').reset_index(drop=True); n = len(part)
    part['p_holm'] = np.minimum(1, np.maximum.accumulate([(n - i) * p for i, p in enumerate(part.p)])); cmp.append(part)
cmp = pd.concat(cmp, ignore_index=True); cmp.to_csv(f'{OUT}/iou_caja_pareada.csv', index=False)
print(cmp.round(4).to_string(index=False))

## Subgrupos: grupo dentario, vista y retractor (siempre con n)

In [ ]:
sub = []
for col, lab in (('grupo', 'grupo dentario'), ('view', 'vista'), ('condition', 'retractor')):
    for (m, v), d in T.groupby(['model', col]):
        sub.append(dict(variable=lab, nivel=v, modelo=LABEL[m], n_fotos=d.image_id.nunique(), n_ninos=d.child_id.nunique(), **mets(d),
                        iou_media_S1=d.iou_S1.mean()))
sub = pd.DataFrame(sub)
sub['_o'] = sub.nivel.map({g: i for i, g in enumerate(GRUPOS)}).fillna(99); sub = sub.sort_values(['variable', '_o', 'nivel', 'modelo']).drop(columns='_o')
sub.to_csv(f'{OUT}/iou_caja_subgrupos.csv', index=False)
for lab in ('grupo dentario', 'vista', 'retractor'):
    s = sub[sub.variable == lab]
    tab = s.pivot_table(index='nivel', columns='modelo', values=['iou_media', 'p075'], sort=False)
    tab[('n', 'ref')] = s.groupby('nivel', sort=False).n_ref.first(); tab[('n', 'fotos')] = s.groupby('nivel', sort=False).n_fotos.first()
    print(f'\n{lab}:'); print(tab.round(3).to_string())

## Error de tamaño: área de caja predicha / área de caja de referencia (entre los encontrados)
> 1 = caja agrandada (p. ej., fragmentos sueltos de la clase lejos del diente); < 1 = caja achicada. Se informa también con la caja del componente conexo mayor (S1).

In [ ]:
tam = []
for m in MODELS:
    d = T[(T.model == m) & T.found]
    for col, esc in (('razon_area', 'principal'), ('razon_area_S1', 'S1')):
        r = d[col].to_numpy()
        tam.append(dict(modelo=LABEL[m], escenario=esc, n=len(r), mediana=np.median(r), p10=np.percentile(r, 10), p90=np.percentile(r, 90),
                        pct_mayor_1_25=(r > 1.25).mean(), pct_menor_0_8=(r < 0.8).mean(), pct_mayor_2=(r > 2).mean()))
    tam[-2]['pct_con_mas_de_1_componente'] = (d.n_comp > 1).mean()
tam = pd.DataFrame(tam); tam.to_csv(f'{OUT}/iou_caja_tamano.csv', index=False); print(tam.round(4).to_string(index=False))

## Figura: distribución del IoU de caja entre los encontrados (ECDF)
Panel izquierdo, rango completo; panel derecho, zoom en 0,6–1. Líneas de referencia en 0,5, 0,75 y 0,9.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)
for ax, lo in zip(axs, (0.0, 0.6)):
    for m in MODELS:
        x = np.sort(T[(T.model == m) & T.found].iou.to_numpy()); y = np.arange(1, len(x) + 1) / len(x)
        r = res.set_index('modelo').loc[LABEL[m]]
        ax.step(x, y, where='post', color=COL[m], lw=1.6, label=f"{LABEL[m]} (mediana {r.iou_mediana_enc:.3f}; n = {int(r.n_encontrados)})")
    for t in THRS: ax.axvline(t, color='0.6', lw=0.8, ls='--', zorder=0)
    ax.set_xlim(lo, 1.0); ax.set_ylim(0, 1.0); ax.grid(alpha=.25); ax.set_xlabel('IoU de caja (predicha vs referencia)')
axs[0].set_ylabel('Proporción acumulada de dientes encontrados'); axs[0].set_title('Rango completo', fontsize=10); axs[1].set_title('Zoom 0,6–1', fontsize=10)
axs[0].legend(loc='upper left', fontsize=7.5, frameon=False)
fig.tight_layout(); fig.savefig(f'{OUT}/iou_caja_distribucion.pdf'); fig.savefig(f'{OUT}/iou_caja_distribucion.png', dpi=200); plt.show()

## Sensibilidad
- **S1, componente conexo mayor:** la caja predicha usa solo el componente conexo más grande de la clase (los 4 modelos; YOLO sobre su máscara pintada). Mismo conjunto de encontrados que la regla principal.
- **S2, YOLO crudo** (si el caché de `map-segmentadores` trae las máscaras de instancias en RLE): por cada FDI de la referencia, la caja de la instancia de **mayor confianza** con conf ≥ 0,25 (el umbral del benchmark) y máscara ≥ 50 px, sin pintar. **S2b**: lo mismo sin umbral de confianza (todas las instancias guardadas, conf ≥ 0,001), solo informativo.
- En S2/S2b los semánticos mantienen la regla principal. Se reporta el IoU medio de punta a punta y la proporción ≥ 0,75 con IC, y el orden de los modelos en cada escenario.

In [ ]:
def rle_counts(s):
    """Decodifica el string comprimido de COCO (igual que rleFrString de pycocotools)."""
    if isinstance(s, list): return s
    if isinstance(s, str): s = s.encode()
    cnts, p = [], 0
    while p < len(s):
        x, k, more = 0, 0, 1
        while more:
            c = s[p] - 48; x |= (c & 0x1f) << 5 * k; more = c & 0x20; p += 1; k += 1
            if not more and (c & 0x10): x |= -1 << 5 * k
        if len(cnts) > 2: x += cnts[-2]
        cnts.append(x)
    return cnts
def rle_box(r):
    """Caja semiabierta (y0, x0, y1, x1) y área de una máscara RLE de COCO (orden por columnas)."""
    h = r['size'][0]; cnts = rle_counts(r['counts']); pos = 0; y0 = x0 = 10 ** 9; y1 = x1 = -1; a = 0
    for i, c in enumerate(cnts):
        if i % 2 and c > 0:
            s, e = pos, pos + c - 1; cs, ce = s // h, e // h; a += c
            x0, x1 = min(x0, cs), max(x1, ce)
            if cs == ce: y0, y1 = min(y0, s % h), max(y1, e % h)
            else: y0, y1 = 0, h - 1
        pos += c
    return ((y0, x0, y1 + 1, x1 + 1) if a else NA), a

S2_OK, s2_msg = False, ''
cache = find('instancias.pkl.gz')
if cache is None: s2_msg = 'no está instancias.pkl.gz entre los inputs: S2 se omite'
else:
    INST = pickle.load(gzip.open(cache)); Y = INST.get('yolo11m_seg', {}); del INST
    ex_ = next((x for r in Y.values() for x in r.get('main', [])), None)
    if len(Y) != len(man) or ex_ is None or len(ex_) != 3 or not (isinstance(ex_[2], dict) and 'counts' in ex_[2]):
        s2_msg = f'el caché no trae máscaras RLE de las instancias crudas de YOLO (fotos {len(Y)}, ejemplo {type(ex_[2]) if ex_ else None}): S2 se omite'
    else:
        S2_OK = True; s2_msg = f'caché OK: {sum(len(r["main"]) for r in Y.values())} instancias crudas de YOLO (conf ≥ 0,001) en {len(Y)} fotos'
        try:   # control cruzado del decodificador con pycocotools, si está instalado
            from pycocotools import mask as MU
            smp_ = [x[2] for r in list(Y.values())[:20] for x in r['main'][:5]]
            ok_ = all(abs(rle_box(q)[1] - int(MU.area(q))) == 0 and np.allclose(np.r_[rle_box(q)[0][1], rle_box(q)[0][0], rle_box(q)[0][3] - rle_box(q)[0][1], rle_box(q)[0][2] - rle_box(q)[0][0]], MU.toBbox(q)) for q in smp_)
            s2_msg += f' | decodificador RLE vs pycocotools en {len(smp_)} máscaras: {"coincide" if ok_ else "NO COINCIDE"}'; assert ok_
        except ImportError: s2_msg += ' | pycocotools no instalado: sin control cruzado del decodificador'
print(s2_msg)

if S2_OK:
    TY = T[T.model == 'yolo11m_seg']; s2 = {th: [] for th in ('S2', 'S2b')}
    for j, f in zip(TY.image_id, TY.fdi):
        L = [x for x in Y[j]['main'] if x[0] == f]
        for th, cmin in (('S2', S2_CONF), ('S2b', 0.0)):
            best = None
            for c_, s_, q in sorted(L, key=lambda x: -x[1]):
                if s_ < cmin: break
                b, a = rle_box(q)
                if a >= MIN_PX: best = b; break
            s2[th].append(best)
    for th, L in s2.items():
        fnd = np.array([b is not None for b in L]); Pb = np.array([b if b is not None else NA for b in L], np.int64)
        Gy = TY[['gy0', 'gx0', 'gy1', 'gx1']].to_numpy(np.int64)
        T.loc[TY.index, f'found_{th}'] = fnd; T.loc[TY.index, f'iou_{th}'] = np.where(fnd, box_iou(Gy, Pb), 0.0)
        T.loc[TY.index, f'razon_area_{th}'] = np.where(fnd, area(Pb) / area(Gy), np.nan)
    del Y

sen = []
SCEN = [('principal', 'iou', 'found'), ('S1', 'iou_S1', 'found')] + ([('S2', 'iou_S2', 'found_S2'), ('S2b', 'iou_S2b', 'found_S2b')] if S2_OK else [])
for esc, ci, cf in SCEN:
    for m in MODELS:
        d = T[T.model == m]; cambia = esc in ('principal', 'S1') or m == 'yolo11m_seg'
        ic, fc = (ci, cf) if cambia else ('iou', 'found')
        d = d.assign(**{fc: d[fc].astype(bool)}); pt = mets(d, ic, fc); bs = boot(d, ic, fc)
        sen.append(dict(escenario=esc, modelo=LABEL[m], cambia=cambia, n_encontrados=pt['n_encontrados'],
                        iou_media=pt['iou_media'], iou_media_lo=bs.iou_media.quantile(.025), iou_media_hi=bs.iou_media.quantile(.975),
                        p075=pt['p075'], p075_lo=bs.p075.quantile(.025), p075_hi=bs.p075.quantile(.975),
                        iou_mediana_enc=pt['iou_mediana_enc'], razon_area_mediana=d.loc[d[fc], 'razon_area' + ('' if ic == 'iou' else ic[3:])].median()))
sen = pd.DataFrame(sen)
for x in ('iou_media', 'p075'): sen[f'orden_{x}'] = sen.groupby('escenario')[x].rank(ascending=False, method='min').astype(int)
sen.to_csv(f'{OUT}/iou_caja_sensibilidad.csv', index=False)
print(sen.round(4).to_string(index=False))
base = sen[sen.escenario == 'principal'].set_index('modelo')
for esc in sen.escenario.unique():
    s = sen[sen.escenario == esc].set_index('modelo')
    print(f'{esc:10s} orden IoU medio: {" > ".join(s.sort_values("iou_media", ascending=False).index)}'
          f'  | ¿cambia vs principal? IoU medio {"sí" if (s.orden_iou_media != base.orden_iou_media).any() else "no"}, p075 {"sí" if (s.orden_p075 != base.orden_p075).any() else "no"}')

## Tabla por diente, comparación informativa con la evaluación encadenada y lista de salidas
En la evaluación encadenada (pipeline-icdas-v4) la mediana del IoU de SegFormer-B2 contra la **caja manual** fue 0,84. Aquí la referencia es la caja de la máscara FDI, así que no tiene por qué coincidir: solo se reporta.

In [ ]:
cols = ['model', 'image_id', 'child_id', 'fold', 'view', 'condition', 'fdi', 'grupo', 'gt_px', 'gy0', 'gx0', 'gy1', 'gx1', 'gt_area_caja',
        'pred_px', 'found', 'py0', 'px0', 'py1', 'px1', 'iou', 'razon_area', 'n_comp', 'ly0', 'lx0', 'ly1', 'lx1', 'iou_S1', 'razon_area_S1']
cols += [c for c in ('found_S2', 'iou_S2', 'razon_area_S2', 'found_S2b', 'iou_S2b', 'razon_area_S2b') if c in T]
T[cols].assign(modelo=T.model.map(LABEL)).to_csv(f'{OUT}/iou_caja_por_diente.csv', index=False)
sg = T[(T.model == 'segformer_b2') & T.found].iou
print(f'SegFormer-B2, IoU de caja vs caja de la máscara: mediana {sg.median():.3f} (entre encontrados, n = {len(sg)}); '
      f'de punta a punta {T[T.model == "segformer_b2"].iou.median():.3f} | encadenada vs caja manual: 0,84 (solo informativo)')
chk2 = chk.copy(); chk2['S2'] = s2_msg; chk2.to_csv(f'{OUT}/iou_caja_chequeos.csv', index=False)
print('\ntiempo total', round((time.time() - T0) / 60, 1), 'min'); print(sorted(os.listdir(OUT)))